# Fall Detection Le2i Frame Check

This notebook does one thing only: select one Le2i video, sample frames from it, pick one frame, run the Hugging Face fall detector on that frame, and output `Fall` or `Normal` plus one annotated image.


## Section 1: Environment + Imports


In [ ]:
# Install required packages
!pip install -q ultralytics huggingface_hub opencv-python pillow matplotlib av

print("Packages installed successfully")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from huggingface_hub import hf_hub_download, snapshot_download
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CUDA_DEVICE_COUNT = torch.cuda.device_count() if DEVICE == "cuda" else 0
BASE = Path("/kaggle/working")
INPUT_BASE = Path("/kaggle/input")
DOWNLOADS_DIR = BASE / "downloads_hf"
DEMO_INPUTS_DIR = BASE / "demo_inputs"
RESULTS_DIR = BASE / "results_hf"
SAMPLED_FRAMES_DIR = DEMO_INPUTS_DIR / "sampled_frames"

for path in [DOWNLOADS_DIR, DEMO_INPUTS_DIR, RESULTS_DIR, SAMPLED_FRAMES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"OpenCV: {cv2.__version__}")
if DEVICE == "cuda":
    print(f"Using GPU: cuda:0 ({torch.cuda.get_device_name(0)})")
else:
    print("GPU not detected. Notebook will run on CPU.")


## Section 2: Config + Model Load


In [ ]:
VIDEO_INPUT_PATH = None
LE2I_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/tuyenldvn/falldataset-imvia/Office"),
    Path("/kaggle/input/datasets/tuyenldvn/falldataset-imvia"),
    Path("/kaggle/input/falldataset-imvia/Office"),
    Path("/kaggle/input/falldataset-imvia"),
]

YOLO_MODEL_ID = "melihuzunoglu/human-fall-detection"
YOLO_CONFIDENCE_THRESHOLD = 0.35
YOLO_IOU_THRESHOLD = 0.45
FRAME_SAMPLE_COUNT = 12
FRAME_SAMPLING_TIMEOUT_SECONDS = 30
MAX_VIDEO_SCAN_RESULTS = 256
MAX_VIDEO_SCAN_DEPTH = 4

GPU_INDEX = 0
if DEVICE == "cuda" and GPU_INDEX >= max(1, CUDA_DEVICE_COUNT):
    raise ValueError(f"GPU_INDEX={GPU_INDEX} but only {CUDA_DEVICE_COUNT} CUDA devices are visible")

YOLO_DEVICE = GPU_INDEX if DEVICE == "cuda" else "cpu"
YOLO_DEVICE_LABEL = f"cuda:{GPU_INDEX}" if DEVICE == "cuda" else "cpu"
YOLO_HALF = DEVICE == "cuda"
YOLO_IMGSZ = 640 if DEVICE == "cuda" else 416

SELECTED_VIDEO_PATH = None
SELECTED_FRAME_PATH = None
FRAME_RESULT_LABEL = None
FRAME_RESULT_CONFIDENCE = 0.0
ANNOTATED_FRAME_PATH = RESULTS_DIR / "annotated_frame.jpg"

print(f"YOLO model id: {YOLO_MODEL_ID}")
print(f"YOLO device:   {YOLO_DEVICE_LABEL}")
print(f"YOLO imgsz:    {YOLO_IMGSZ}")
print(f"Frame samples: {FRAME_SAMPLE_COUNT}")

def load_yolo_hf_model(repo_id):
    local_dir = DOWNLOADS_DIR / "hf_yolo_model"
    local_dir.mkdir(parents=True, exist_ok=True)
    try:
        weight_path = Path(
            hf_hub_download(
                repo_id=repo_id,
                filename="best.pt",
                repo_type="model",
                local_dir=str(local_dir),
            )
        )
    except Exception:
        repo_dir = Path(
            snapshot_download(
                repo_id=repo_id,
                repo_type="model",
                local_dir=str(local_dir),
            )
        )
        candidates = sorted(repo_dir.rglob("best.pt")) + sorted(repo_dir.rglob("*.pt"))
        if not candidates:
            raise FileNotFoundError(f"No .pt weights found in {repo_dir}")
        weight_path = candidates[0]
    model = YOLO(str(weight_path))
    names = model.names if isinstance(model.names, dict) else dict(enumerate(model.names))
    fall_class_ids = [
        int(cls_id)
        for cls_id, name in names.items()
        if any(token in str(name).lower() for token in ["fall", "fallen", "lying"])
    ]
    return model, weight_path, names, fall_class_ids

yolo_model, YOLO_WEIGHT_PATH, YOLO_CLASS_NAMES, FALL_CLASS_IDS = load_yolo_hf_model(YOLO_MODEL_ID)
print(f"YOLO weights:  {YOLO_WEIGHT_PATH}")
print(f"YOLO classes:  {YOLO_CLASS_NAMES}")


## Section 3: Le2i Video Selection + Frame Sampling


In [ ]:
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv", ".wmv"}
SAMPLED_FRAME_RECORDS = []

def resolve_le2i_input_root():
    for candidate in LE2I_INPUT_CANDIDATES:
        if candidate.exists():
            return candidate
    return None

def iter_video_files_limited(root_dir, max_depth=MAX_VIDEO_SCAN_DEPTH, max_results=MAX_VIDEO_SCAN_RESULTS):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        return
    seen = set()
    yielded = 0
    root_depth = len(root_dir.parts)
    for current_root, dirnames, filenames in os.walk(root_dir):
        current_path = Path(current_root)
        depth = len(current_path.parts) - root_depth
        if depth >= max_depth:
            dirnames[:] = []
        dirnames[:] = sorted(dirnames)[:64]
        for filename in sorted(filenames):
            suffix = Path(filename).suffix.lower()
            if suffix not in VIDEO_EXTS:
                continue
            candidate = current_path / filename
            candidate_key = str(candidate)
            if candidate_key in seen:
                continue
            seen.add(candidate_key)
            yield candidate
            yielded += 1
            if yielded >= max_results:
                return

def pick_best_video_from_dir(search_dir):
    best_fall = None
    best_other = None
    for candidate in iter_video_files_limited(search_dir):
        key = (len(str(candidate)), str(candidate))
        if "fall" in str(candidate).lower():
            if best_fall is None or key < best_fall[0]:
                best_fall = (key, candidate)
        else:
            if best_other is None or key < best_other[0]:
                best_other = (key, candidate)
    if best_fall is not None:
        return best_fall[1]
    if best_other is not None:
        return best_other[1]
    return None

def choose_le2i_demo_from_input_root(root_dir):
    root_dir = Path(root_dir)
    if root_dir.name.lower() == "office":
        chosen = pick_best_video_from_dir(root_dir)
        if chosen is not None:
            return chosen, "Le2i/Office"
        return None, None
    preference_dirs = [
        ("Le2i/Office", root_dir / "Office"),
        ("Le2i/Lecture_room", root_dir / "Lecture_room"),
        ("Le2i/Coffee_room_01", root_dir / "Coffee_room_01"),
        ("Le2i/Coffee_room_02", root_dir / "Coffee_room_02"),
        ("Le2i/Home_01", root_dir / "Home_01"),
        ("Le2i/Home_02", root_dir / "Home_02"),
    ]
    for subset_label, subset_dir in preference_dirs:
        if not subset_dir.exists():
            continue
        chosen = pick_best_video_from_dir(subset_dir)
        if chosen is not None:
            return chosen, subset_label
    return None, None

def select_video_path():
    if VIDEO_INPUT_PATH and Path(VIDEO_INPUT_PATH).exists():
        return Path(VIDEO_INPUT_PATH), "user video", "user supplied"
    mounted_root = resolve_le2i_input_root()
    if mounted_root is None:
        raise FileNotFoundError(
            "No Le2i input found. Set VIDEO_INPUT_PATH or mount one of LE2I_INPUT_CANDIDATES."
        )
    mounted_video, mounted_subset = choose_le2i_demo_from_input_root(mounted_root)
    if mounted_video is None:
        raise FileNotFoundError(f"No video files found under Le2i root: {mounted_root}")
    return Path(mounted_video), "Le2i mounted input", mounted_subset

def extract_sampled_frames_isolated(video_path, output_dir, sample_count=FRAME_SAMPLE_COUNT, timeout_seconds=FRAME_SAMPLING_TIMEOUT_SECONDS):
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    child_code = """
import json
import sys
from pathlib import Path
import av
from PIL import Image

video_path = Path(sys.argv[1])
output_dir = Path(sys.argv[2])
sample_count = max(1, int(sys.argv[3]))

def count_frames(path):
    container = av.open(str(path))
    try:
        total = 0
        for _ in container.decode(video=0):
            total += 1
        return total
    finally:
        container.close()

total_frames = count_frames(video_path)
if total_frames <= 0:
    raise RuntimeError(f'No readable frames found in {video_path}')

if sample_count == 1:
    target_indices = [total_frames // 2]
else:
    target_indices = sorted({
        int(round(i * (total_frames - 1) / (sample_count - 1)))
        for i in range(sample_count)
    })

saved = []
target_set = set(target_indices)
container = av.open(str(video_path))
try:
    for idx, frame in enumerate(container.decode(video=0)):
        if idx not in target_set:
            continue
        rgb = frame.to_ndarray(format='rgb24')
        image_path = output_dir / f'frame_{idx:06d}.jpg'
        Image.fromarray(rgb).save(str(image_path))
        saved.append({'frame_index': idx, 'path': str(image_path)})
        if len(saved) >= len(target_indices):
            break
finally:
    container.close()

if not saved:
    raise RuntimeError(f'Failed to save sampled frames from {video_path}')

print(json.dumps({'total_frames': total_frames, 'frames': saved}))
"""

    cmd = [sys.executable, '-c', child_code, str(video_path), str(output_dir), str(sample_count)]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_seconds, check=False)
    except subprocess.TimeoutExpired as e:
        raise TimeoutError(
            f"Frame sampling timed out after {timeout_seconds}s for {video_path}"
        ) from e
    if result.returncode != 0:
        detail = (result.stderr or result.stdout or '').strip()
        raise RuntimeError(
            f"Frame sampling failed for {video_path} with exit code {result.returncode}: {detail}"
        )
    payload = json.loads((result.stdout or '').strip())
    return payload

SELECTED_VIDEO_PATH, SELECTED_DEMO_SOURCE, SELECTED_DEMO_SUBSET = select_video_path()
frame_sample_payload = extract_sampled_frames_isolated(SELECTED_VIDEO_PATH, SAMPLED_FRAMES_DIR)
SAMPLED_FRAME_RECORDS = frame_sample_payload['frames']
if not SAMPLED_FRAME_RECORDS:
    raise RuntimeError(f"No sampled frames were created for {SELECTED_VIDEO_PATH}")

print(f"Selected video: {SELECTED_VIDEO_PATH}")
print(f"Source:         {SELECTED_DEMO_SOURCE}")
print(f"Subset:         {SELECTED_DEMO_SUBSET}")
print(f"Sampled frames: {len(SAMPLED_FRAME_RECORDS)}")


## Section 4: Single-Frame Inference + Output


In [ ]:
def label_from_class_id(class_id):
    return str(YOLO_CLASS_NAMES.get(int(class_id), f"class_{int(class_id)}"))

def analyze_result(result):
    if result.boxes is None or result.boxes.xyxy is None:
        return {
            'label': 'Normal',
            'confidence': 0.0,
            'detections': [],
        }
    boxes = result.boxes
    confs = boxes.conf.cpu().numpy().tolist() if boxes.conf is not None else []
    classes = boxes.cls.cpu().numpy().tolist() if boxes.cls is not None else []
    detections = []
    for conf, cls_id in zip(confs, classes):
        cls_id = int(cls_id)
        label = label_from_class_id(cls_id)
        is_fall = cls_id in FALL_CLASS_IDS if FALL_CLASS_IDS else any(
            token in label.lower() for token in ['fall', 'fallen', 'lying']
        )
        detections.append({
            'class_id': cls_id,
            'class_label': label,
            'confidence': float(conf),
            'is_fall': bool(is_fall),
        })
    fall_confidences = [d['confidence'] for d in detections if d['is_fall']]
    if fall_confidences:
        return {
            'label': 'Fall',
            'confidence': float(max(fall_confidences)),
            'detections': detections,
        }
    normal_confidence = float(max([d['confidence'] for d in detections], default=0.0))
    return {
        'label': 'Normal',
        'confidence': normal_confidence,
        'detections': detections,
    }

def predict_frame(frame_path):
    results = yolo_model.predict(
        source=str(frame_path),
        conf=YOLO_CONFIDENCE_THRESHOLD,
        iou=YOLO_IOU_THRESHOLD,
        device=YOLO_DEVICE,
        half=YOLO_HALF,
        imgsz=YOLO_IMGSZ,
        verbose=False,
    )
    result = results[0]
    analysis = analyze_result(result)
    return result, analysis

frame_summaries = []
chosen_result = None
chosen_analysis = None
chosen_frame_record = None

for record in SAMPLED_FRAME_RECORDS:
    result, analysis = predict_frame(record['path'])
    summary = {
        'frame_index': int(record['frame_index']),
        'path': record['path'],
        'label': analysis['label'],
        'confidence': float(analysis['confidence']),
    }
    frame_summaries.append(summary)
    if analysis['label'] == 'Fall':
        chosen_result = result
        chosen_analysis = analysis
        chosen_frame_record = record
        break

if chosen_frame_record is None:
    middle_record = SAMPLED_FRAME_RECORDS[len(SAMPLED_FRAME_RECORDS) // 2]
    chosen_result, chosen_analysis = predict_frame(middle_record['path'])
    chosen_frame_record = middle_record

SELECTED_FRAME_PATH = Path(chosen_frame_record['path'])
FRAME_RESULT_LABEL = chosen_analysis['label']
FRAME_RESULT_CONFIDENCE = float(chosen_analysis['confidence'])

annotated = chosen_result.plot()
ANNOTATED_FRAME_PATH.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(ANNOTATED_FRAME_PATH), annotated)

print(f"Selected video path: {SELECTED_VIDEO_PATH}")
print(f"Selected frame path: {SELECTED_FRAME_PATH}")
print(f"Final label:         {FRAME_RESULT_LABEL}")
print(f"Confidence:          {FRAME_RESULT_CONFIDENCE:.4f}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"{FRAME_RESULT_LABEL} ({FRAME_RESULT_CONFIDENCE:.4f})")
plt.axis('off')
plt.show()

print(f"Annotated image saved to: {ANNOTATED_FRAME_PATH}")
